In [31]:
from langchain.document_loaders import PyMuPDFLoader, UnstructuredPDFLoader

def parse_file(file_path):
    try:
        loader = PyMuPDFLoader(file_path)
        documents = loader.load()
        print("✅ Parsed with PyMuPDFLoader")
    except Exception as e:
        print(f"⚠️ PyMuPDF failed: {e} — trying UnstructuredPDFLoader")
        loader = UnstructuredPDFLoader(file_path)
        documents = loader.load()
        print("✅ Parsed with UnstructuredPDFLoader")
    return documents


In [32]:
from email.mime.text import MIMEText
import smtplib
email_tracker = {}  # user_email: count

def send_email(recipient_email, subject):
    body = f"""
    Hi there,
    
    Your document **"Airsight Solution Doc.pdf"** has been successfully parsed and uploaded to Google Drive.
    
    🔗 You can access the file here:
    https://drive.google.com/drive/folders/1lJQnX2hJb2eMBarCEcIPthKUFktSQ6JD
    
    If you have any questions or want to chat with the AI agent based on this file, feel free to reply!
    
    Regards,  
    RAG AI Assistant 🤖"""
    if email_tracker.get(recipient_email, 0) >= 5:
        return "Email limit reached."

    msg = MIMEText(body)
    msg['Subject'] = subject
    msg['From'] = 'vinaykamble289@gmail.com'
    msg['To'] = recipient_email

    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.starttls()
        server.login('vinaykamble289@gmail.com', 'bmie soce blor xygt')
        server.sendmail(msg['From'], [msg['To']], msg.as_string())

    email_tracker[recipient_email] = email_tracker.get(recipient_email, 0) + 1
    return "Email sent successfully."


In [33]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

def setup_rag(documents):
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    
    vectordb = FAISS.from_documents(documents, embedding_model)
    retriever = vectordb.as_retriever()

    model_id = "google/flan-t5-base"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

    pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer, max_length=512)
    llm = HuggingFacePipeline(pipeline=pipe)
    
    rag_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

    return rag_chain


In [38]:
import os

# Initialize agent variable
agent = None

def setup_chatbot(file_path):
    """Setup the RAG chatbot with a file"""
    global agent
    try:
        print(f"Processing file: {file_path}")
        
        # Get user email and send notification
        mail_id = input("Enter email: ").strip()
        documents = parse_file(file_path)

        email_result = send_email(mail_id, subject=f'file processed{file_path}')
        print(email_result)

        # Setup RAG agent
        agent = setup_rag(documents)
        print("🤖 RAG agent initialized!\n")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

        
def chat():
    """Simple chat loop"""
    global agent
    
    if agent is None:
        print("❌ Please setup the chatbot first with setup_chatbot('your_file.pdf')")
        return
    
    print("💬 Chat started! Type 'quit' to exit.\n")
    
    while True:
        query = input("You: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("👋 Goodbye!")
            break
            
        if not query:
            continue
            
        try:
            response = agent.invoke(query)
            print(f"Bot: {response['result']}\n")
            
        except Exception as e:
            print(f"❌ Error: {str(e)}\n")

# Usage example:
print("📇 Minimal RAG Chatbot Test")
print("="*40)
print("1. First run: setup_chatbot('your_file.pdf')")
print("2. Then run: chat()")
print("="*40)

📇 Minimal RAG Chatbot Test
1. First run: setup_chatbot('your_file.pdf')
2. Then run: chat()


In [36]:
setup_chatbot('Airsight Solution Doc.pdf')

Processing file: Airsight Solution Doc.pdf


Enter email:  vinaykamble289@gmail.com


✅ Parsed with PyMuPDFLoader
Email sent successfully.


C:\Users\VINAY\AppData\Local\Temp\ipykernel_17444\2104414175.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Device set to use cpu


🤖 RAG agent initialized!



C:\Users\VINAY\AppData\Local\Temp\ipykernel_17444\2104414175.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [37]:
chat()

💬 Chat started! Type 'quit' to exit.



You:  what is airsight?


Token indices sequence length is longer than the specified maximum sequence length for this model (944 > 512). Running this sequence through the model will result in indexing errors


Bot: AQI app



You:  what is need of airsight


Bot: millions of Indians in rural and semi-urban areas who live in data darkness



You:  quit


👋 Goodbye!
